# Personality Type Prediction – Part 3: Tuning and Tracking

In this notebook I use Hyperopt to find good settings for the Random Forest.
Every try is logged with MLflow, so I can compare them later in the MLflow UI.
At the end I log the best model, so the Streamlit app can load it.

## 1. Setup

Same data cleaning and pipeline as in notebook 02.

In [2]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# load the data
df = pd.read_csv("data.csv")

# data cleaning (same as notebook 01)
item_cols = [c for c in df.columns if c not in ["age", "gender", "hand", "target"]]
df.loc[df["age"] > 100, "age"] = np.nan
df[item_cols] = df[item_cols].replace(0, np.nan)
df = df.drop_duplicates()

# features and target
X = df.drop(columns=["target"])
y = df["target"]

# preprocessing pipeline
preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]), item_cols + ["age"]),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), ["gender", "hand"]),
])

print("Shape:", df.shape)

Shape: (19716, 23)


## 2. MLflow setup

I use an absolute path for the database, like in the course.
The path is created from the project folder, so it also works on other computers.

In [3]:
import mlflow
from pathlib import Path

# absolute path to mlflow.db in the project folder
db_path = Path("mlflow.db").resolve()
mlflow.set_tracking_uri(f"sqlite:///{db_path.as_posix()}")

mlflow.set_experiment("personality-type-prediction")

print("Tracking URI:", mlflow.get_tracking_uri())

c:\AI_Engineering_local\Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/10/25 16:13:13 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/25 16:13:13 INFO mlflow.store.db.utils: Updating database tables
2026/10/25 16:13:16 INFO mlflow.tracking.fluent: Experiment with name 'personality-type-prediction' does not exist. Creating a new experiment.


Tracking URI: sqlite:///C:/AI_Engineering_local/Project/mlflow.db


## 3. Train/test split

I split the data before tuning. The tuning only uses the train data.
The test data is used only one time at the end, for the final check.

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (15772, 22)
Test: (3944, 22)


## 4. Compare models

Before I choose the final model, I compare different models with 5-fold cross-validation on the train data.
`class_weight="balanced"` gives the small classes more weight, so the model cares more about Undercontroller.

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

# XGBoost needs numbers as target, not text
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)

models = {
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=42),
    "LogisticRegression_balanced": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    "RandomForest": RandomForestClassifier(random_state=42, n_jobs=-1),
    "RandomForest_balanced": RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1),
    "XGBoost": XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.1, random_state=42, n_jobs=-1),
}

for name, model in models.items():
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", model),
    ])

    if name == "XGBoost":
        target = y_train_encoded
    else:
        target = y_train

    score = cross_val_score(pipe, X_train, target, cv=5, scoring="f1_macro").mean()

    with mlflow.start_run(run_name=name):
        mlflow.log_param("model", name)
        mlflow.log_metric("cv_f1_macro", score)

    print(f"{name}: F1-macro = {score:.4f}")

LogisticRegression: F1-macro = 0.7679
LogisticRegression_balanced: F1-macro = 0.7838
RandomForest: F1-macro = 0.7430
RandomForest_balanced: F1-macro = 0.7840
XGBoost: F1-macro = 0.7961


## 5. Tuning the best model (XGBoost)

XGBoost has the best F1-macro, so I tune it with Hyperopt.
Every try is evaluated with 5-fold cross-validation on the train data and logged in MLflow.

In [6]:
from hyperopt import fmin, tpe, hp, Trials

def objective(params):
    n_estimators = int(params["n_estimators"])
    max_depth = int(params["max_depth"])
    learning_rate = params["learning_rate"]

    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", XGBClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            learning_rate=learning_rate,
            random_state=42,
            n_jobs=-1,
        )),
    ])

    score = cross_val_score(pipe, X_train, y_train_encoded, cv=5, scoring="f1_macro").mean()

    # log this try in MLflow
    with mlflow.start_run(run_name="xgb-hyperopt"):
        mlflow.log_param("model", "XGBoost")
        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)
        mlflow.log_param("learning_rate", learning_rate)
        mlflow.log_metric("cv_f1_macro", score)

    return -score  # Hyperopt minimizes, so negate the score

space = {
    "n_estimators": hp.quniform("n_estimators", 100, 500, 50),
    "max_depth": hp.quniform("max_depth", 3, 8, 1),
    "learning_rate": hp.loguniform("learning_rate", np.log(0.01), np.log(0.3)),
}

trials = Trials()
best = fmin(
    fn=objective,
    space=space,
    algo=tpe.suggest,
    max_evals=20,
    trials=trials,
    rstate=np.random.default_rng(42),
)

best_params = {
    "n_estimators": int(best["n_estimators"]),
    "max_depth": int(best["max_depth"]),
    "learning_rate": float(best["learning_rate"]),
}
best_score = -min(trials.losses())

print("Best params:", best_params)
print("Best CV F1-macro:", round(best_score, 4))

100%|██████████| 20/20 [02:04<00:00,  6.23s/trial, best loss: -0.7936945984692632]
Best params: {'n_estimators': 450, 'max_depth': 7, 'learning_rate': 0.1609905304007864}
Best CV F1-macro: 0.7937


**Notes:**
- XGBoost with default settings (300 trees, max_depth 4, learning rate 0.1): F1-macro 0.7961
- Best tuned XGBoost (450 trees, max_depth 7, learning rate 0.16): F1-macro 0.7937
- The tuning did not make the model better. The difference is very small, smaller than the difference between folds.
- So I use XGBoost with the default settings as final model. It is also smaller and faster.

## 6. Final model

I train XGBoost with the default settings on all train data.
Then I check it one time on the test data and log the model in MLflow.

In [7]:
from sklearn.metrics import f1_score, accuracy_score, classification_report

final_params = {"n_estimators": 300, "max_depth": 4, "learning_rate": 0.1}

final_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(**final_params, random_state=42, n_jobs=-1)),
])

final_model.fit(X_train, y_train_encoded)

# predict on test data and translate the numbers back to type names
predictions = label_encoder.inverse_transform(final_model.predict(X_test))

test_f1 = f1_score(y_test, predictions, average="macro")
test_accuracy = accuracy_score(y_test, predictions)

with mlflow.start_run(run_name="xgb-final") as run:
    mlflow.log_param("model", "XGBoost")
    mlflow.log_params(final_params)
    mlflow.log_param("classes", list(label_encoder.classes_))
    mlflow.log_metric("test_f1_macro", test_f1)
    mlflow.log_metric("test_accuracy", test_accuracy)
    mlflow.sklearn.log_model(
        final_model,
        name="model",
        input_example=X_train.head(),
        serialization_format="cloudpickle",
    )

print("Run ID:", run.info.run_id)
print("Test F1-macro:", round(test_f1, 4))
print("Test accuracy:", round(test_accuracy, 4))
print()
print(classification_report(y_test, predictions))

2026/10/25 18:29:47 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
c:\AI_Engineering_local\Project\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values

Run ID: 76c771b2b95744a982eaf3bb82871102
Test F1-macro: 0.8026
Test accuracy: 0.8656

                 precision    recall  f1-score   support

       Moderate       0.83      0.90      0.86      1690
 Overcontroller       0.86      0.86      0.86       566
      Resilient       0.97      0.99      0.98      1233
Undercontroller       0.62      0.42      0.50       455

       accuracy                           0.87      3944
      macro avg       0.82      0.79      0.80      3944
   weighted avg       0.86      0.87      0.86      3944



**Notes:**
- Final model: XGBoost (300 trees, max_depth 4, learning rate 0.1)
- Test F1-macro: 0.803 (cross-validation was 0.796, so the model is stable)
- Compared to the Random Forest from notebook 02 (F1-macro 0.75), XGBoost is much better.
- Undercontroller is still the hardest class, but recall went up from 0.21 to 0.42.
- Run ID of the final model: 76c771b2b95744a982eaf3bb82871102